# Conditional Edge

## 조건부 Edge(`Conditional Edge`) 사용해 분기 만들기
* `Conditional Edge`를 이용해 상황에 따라 다른 노드로 연결되도록 구현할 수 있습니다.

In [8]:
from typing import Literal, Annotated
from pydantic import BaseModel, Field

from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage, HumanMessage

import time

class AgentState(BaseModel):
    # Annotated[타입, 메타데이터] -> 해당 변수에 '메타데이터'에 해당하는 특성(기능)을 부여함
    # add_messages -> message에 적용하는 메타데이터, Node에서 history에 추가할 항목만 반환해도 알아서 리스트에 추가해 줌
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list) # 맨 처음 생성 시 빈 리스트로 시작
    last_player: Literal['A', 'B']
    number: int
    end: bool

In [15]:
def node_A(state: AgentState) -> dict:
    num = state.number
    new_num = num * 2
    time.sleep(1)
    return {
        'chat_history': [HumanMessage(name='A', content=f"숫자를 {num}에서 {new_num}(으)로 바꿈")],
        'number': new_num,
        'last_player': 'A'
    }

In [16]:
def node_B(state: AgentState) -> dict:
    num = state.number
    new_num = num - 1
    time.sleep(1)
    return {
        'chat_history': [HumanMessage(name='B', content=f"숫자를 {num}에서 {new_num}(으)로 변경")],
        'number': new_num,
        'last_player': 'B'
    }

In [17]:
def node_judge(state: AgentState) -> dict:
    end = True if state.number > 100 else False
    return {
        'chat_history': [HumanMessage(name='judge', content=f"최종 숫자 {state.number}(으)로 종료합니다" if end else "계속 진행하세요")],
        'end': end
    }

In [18]:
from langgraph.graph import START, END, StateGraph

workflow = StateGraph(AgentState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('judge', node_judge)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'judge')
workflow.add_edge('B', 'judge')

## `route` 만들고 `conditional edge`에 부여하기
* `route`는 조건에 따라서 달라지는 목적지 `node`명 을 반환하는 함수입니다
* `workflow.add_conditional_edges`함수에 `route`를 적용하면, route가 반환한 `node`명으로 라우팅됩니다.

In [19]:
def judge_route(state: AgentState):
    # end가 True면 END로
    if state.end:
        return END
    elif state.last_player == 'B':
        return 'A'
    else:
        return 'B'

workflow.add_conditional_edges(
    'judge', 
    judge_route
)

In [22]:
app = workflow.compile()

init_input = AgentState(chat_history=[], last_player='A', number=1, end=False)

for chunk in app.stream(init_input):
    print(chunk)

{'A': {'chat_history': [HumanMessage(content='숫자를 1에서 2(으)로 바꿈', additional_kwargs={}, response_metadata={}, name='A', id='480ea6f8-cf3a-48e3-bf64-faf14f606968')], 'number': 2, 'last_player': 'A'}}
{'judge': {'chat_history': [HumanMessage(content='계속 진행하세요', additional_kwargs={}, response_metadata={}, name='judge', id='1c80a264-d5f6-4dca-a87a-ed4bf13da351')], 'end': False}}
{'B': {'chat_history': [HumanMessage(content='숫자를 2에서 1(으)로 변경', additional_kwargs={}, response_metadata={}, name='B', id='9464c0b4-fbec-4506-9047-14012dc60bd7')], 'number': 1, 'last_player': 'B'}}
{'judge': {'chat_history': [HumanMessage(content='계속 진행하세요', additional_kwargs={}, response_metadata={}, name='judge', id='d9c2017f-b843-4ba9-a4d0-061b5c51cf28')], 'end': False}}
{'A': {'chat_history': [HumanMessage(content='숫자를 1에서 2(으)로 바꿈', additional_kwargs={}, response_metadata={}, name='A', id='7bc6bf0c-7211-45d2-ad9a-28469f8f9e03')], 'number': 2, 'last_player': 'A'}}
{'judge': {'chat_history': [HumanMessage(content

KeyboardInterrupt: 

## 토마토 게임 만들기
* ai와 토마토 게임을 진행할 수 있는 LangGraph 기반 게임을 만들어 봅시다
* 규칙: 두 명이 번갈아가면서 '토마토'를 한글자씩 말합니다. 올바르게 말하지 못하면 패배합니다.

In [43]:
from pydantic import BaseModel, Field
from typing import Literal, Optional

class TomatoGameState(BaseModel):
    tomato_index: int = Field(ge=0, le=2, description="토마토 중 어떤 글자인지 index")
    correct: bool = True
    player_input: Optional[str] = ''

In [44]:
import time
def user_node(state: TomatoGameState) -> dict:
    user_input = input("당신의 대답: ")
    print(f"사용자: {user_input}")
    return {
        'player_input': user_input
    }

def ai_node(state: TomatoGameState) -> dict:
    time.sleep(1)
    tomato = ('토', '마', '토')
    print(f"ai답변: {tomato[state.tomato_index]}")
    return {
        'tomato_index': (state.tomato_index + 1) % 3
    }

def judge_node(state: TomatoGameState) -> dict:
    tomato = ('토', '마', '토')
    if state.player_input == tomato[state.tomato_index]:
        return {
            'tomato_index': (state.tomato_index + 1) % 3,
            'correct': True
        }
    else:
        print("당신은 틀렸습니다. 큭큭")
        print(f"원래 정답: '{tomato[state.tomato_index]}' 당신의 답: '{state.player_input}'")
        return {
            'correct': False
        }

In [45]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(TomatoGameState)

workflow.add_node("user", user_node)
workflow.add_node("ai", ai_node)
workflow.add_node("judge", judge_node)

workflow.add_edge(START, "user")
workflow.add_edge("user", "judge")
workflow.add_edge("ai", "user")

def tomato_router(state: TomatoGameState):
    if state.correct:
        return 'ai'
    else:
        return END

workflow.add_conditional_edges("judge", tomato_router)

app = workflow.compile()

In [46]:
init_state = TomatoGameState(tomato_index=0, correct=True)

for chunk in app.stream(init_state):
    pass

사용자: 토
ai답변: 마
사용자: 토
ai답변: 토
사용자: 마
ai답변: 토
사용자: 마
당신은 틀렸습니다. 큭큭
원래 정답: '토' 당신의 답: '마'


### 팬아웃(Fan-Out) 구현하기
* 하나의 노드에서 여러 노드로 갈라지고(팬아웃) 다시 한 노드로 흐름을 모을 (팬인) 수 있습니다

In [47]:
from pydantic import BaseModel
from typing import Optional, Any

class FanOutState(BaseModel):
    value1: Optional[Any] = None
    value2: Optional[Any] = None

In [48]:
import time

def node_A(state: FanOutState) -> dict:
    print("A 노드에서 출발합니다.")
    return {"from_node": "A"}

def node_B(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"B 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value1": state.value1 + 1}

def node_C(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": state.value2 + 1}

In [49]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')

app = workflow.compile()

In [50]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/3)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/3)C 노드 실행 중... (2/3)

C 노드 실행 중... (3/3)
B 노드 실행 중... (3/3)


{'value1': 1, 'value2': 1}

## 팬인(Fan-In) 구현하기
* `add_edge` 함수의 출발점은 여러 노드의 배열도 입력받을 수 있습니다.

In [51]:
def node_D(state: FanOutState):
    for i in range(1, 4):
        print(f"D 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": state.value1 + 1}

In [52]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [53]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/3)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/3)C 노드 실행 중... (2/3)

B 노드 실행 중... (3/3)C 노드 실행 중... (3/3)

D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)


{'value1': 1, 'value2': 2}

### Q. 팬인 시 노드 종료 시점이 다르다면?

In [61]:
import time

def node_A(state: FanOutState) -> dict:
    print("A 노드에서 출발합니다.")
    return {"from_node": "A"}

def node_B(state: FanOutState) -> dict: # 두 노드의 실행 시간을 다르게 하고 결과를 확인해보세요.
    value = state.value1
    for i in range (1, 6):
        print(f"노드 B에서 처리 중... {i}/5")
        time.sleep(value + 0.5)
    return {
        'value1': value + 1
    }

def node_C(state: FanOutState) -> dict:
    for i in range(1, 4):
        print(f"노드 C에서 처리 중... {i}/3")
        time.sleep(1)
    return {
        'value2': state.value2 + 1
    }

In [62]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [64]:
init_state = FanOutState(value1=1, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
노드 B에서 처리 중... 1/5
노드 C에서 처리 중... 1/3
노드 C에서 처리 중... 2/3
노드 B에서 처리 중... 2/5
노드 C에서 처리 중... 3/3
노드 B에서 처리 중... 3/5
노드 B에서 처리 중... 4/5
노드 B에서 처리 중... 5/5
D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)


{'value1': 2, 'value2': 3}

# `Reducer` 활용하기

In [65]:
import time

def node_A(state: FanOutState) -> dict:
    print("A 노드에서 출발합니다.")
    return {"from_node": "A"}

def node_B(state: FanOutState) -> dict:
    for i in range(1, 6):
        print(f"B 노드 실행 중... ({i}/5)")
        time.sleep(1)
    return {"value1": state.value1 + 1}

def node_C(state: FanOutState) -> dict: # B, C 노드 모두에서 value1 값을 변경하는 경우
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value1": state.value1 + 1}

    

In [66]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(FanOutState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [67]:
init_state = FanOutState(value1=0, value2=0)
result = app.invoke(init_state) # reducer가 없으면 오류 발생
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
C 노드 실행 중... (1/3)
C 노드 실행 중... (2/3)
B 노드 실행 중... (2/5)
C 노드 실행 중... (3/3)
B 노드 실행 중... (3/5)
B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)


InvalidUpdateError: At key 'value1': Can receive only one value per step. Use an Annotated key to handle multiple values.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_CONCURRENT_GRAPH_UPDATE

In [68]:
from typing import Annotated

def numberAddReducer(left: int, right: int) -> int: # left: 기존 State, right: return 받은 state
    return left + right # return 받은 값으로 대체하지 않고, 기존 값에 return 받은 값을 더함

class ReducerState(BaseModel):
    value1: Annotated[Optional[Any], numberAddReducer] = None
    value2: Optional[Any] = None

In [70]:
def node_B(state: ReducerState) -> dict: # state 정의도 모두 변경
    for i in range(1, 6):
        print(f"B 노드 실행 중... ({i}/5)")
        time.sleep(1)
    return {"value1": 1} # 새로 더해줄 값만 반환하면 됨

def node_C(state: ReducerState) -> dict: # B, C 노드 모두에서 value1 값을 변경하는 경우
    for i in range(1, 4):
        print(f"C 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value1": 1}

def node_D(state: ReducerState):
    for i in range(1, 4):
        print(f"D 노드 실행 중... ({i}/3)")
        time.sleep(1)
    return {"value2": state.value2 + 1}


In [71]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ReducerState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')
workflow.add_edge('A', 'B')
workflow.add_edge('A', 'C')
workflow.add_edge(['B', 'C'], 'D')
workflow.add_edge('D', END)

app = workflow.compile()

In [72]:
init_state = ReducerState(value1=0, value2=0)
result = app.invoke(init_state) # reducer가 없으면 오류 발생
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/5)C 노드 실행 중... (2/3)

C 노드 실행 중... (3/3)B 노드 실행 중... (3/5)

B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)
D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)


{'value1': 2, 'value2': 1}

### 동적 팬아웃 구현하기
* `Conditional Edge`를 이용하면 Fan-Out을 동적으로 설정할 수 있습니다
* Fan-Out을 설정하려면, 목적지 노드 이름을 리스트로 모두 전달하면 됩니다.

In [73]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ReducerState)

workflow.add_node('A', node_A)
workflow.add_node('B', node_B)
workflow.add_node('C', node_C)
workflow.add_node('D', node_D)

workflow.add_edge(START, 'A')

def fanout_router(state: ReducerState):
    if state.value2 > 0:
        return 'B'
    else:
        return ['B', 'C']
    
workflow.add_conditional_edges('A', fanout_router)

workflow.add_edge(['B', 'C'], 'D')

def end_router(state: ReducerState):
    if state.value1 > 2:
        return END
    else:
        return 'A'

workflow.add_conditional_edges('D', end_router)


app = workflow.compile()

In [74]:
init_state = ReducerState(value1=0, value2=0)
result = app.invoke(init_state)
result

A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
C 노드 실행 중... (1/3)
B 노드 실행 중... (2/5)
C 노드 실행 중... (2/3)
C 노드 실행 중... (3/3)
B 노드 실행 중... (3/5)
B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)
D 노드 실행 중... (1/3)
D 노드 실행 중... (2/3)
D 노드 실행 중... (3/3)
A 노드에서 출발합니다.
B 노드 실행 중... (1/5)
B 노드 실행 중... (2/5)
B 노드 실행 중... (3/5)
B 노드 실행 중... (4/5)
B 노드 실행 중... (5/5)


{'value1': 3, 'value2': 1}

### 라우팅 심화 문제
* Fan-Out과 Reducing, 조건부 Edge를 적극적으로 활용해 음식점 운영을 최적화해봅시다 (괄호 안의 숫자는 소요시간(초))
```
burger: patty(5), bun(3) 각각 필요 -> burgerSet(2)
fries: fry(3) -> friesSet(1)
beverage: beverage(2)

세 가지가 모두 준비되면 종료
```

In [99]:
from pydantic import BaseModel
from typing import List, Annotated, Literal, Optional

def menuReducer(left:List[str], right:List[str]) -> List[str]:
    return left + right

class RestaurantState(BaseModel):
    menu_ordered: List[Literal['burger', 'fries', 'beverage']]
    menu_complete: Annotated[Optional[List[Literal['burger', 'fries', 'beverage']]], menuReducer] = []


In [100]:
import time

# 각종 그래프 노드 실행 및 상태 갱신 통합: 반환값으로 state의 menu_complete를 적절히 업데이트하도록 구성
def g_burger_start(state: RestaurantState) -> dict:
    print("버거 준비 시작")
    return {}

def g_fries_start(state: RestaurantState) -> dict:
    print("감자튀김 준비 시작")
    return {}

def g_patty(state: RestaurantState) -> dict:
    print("패티 조리 시작")
    time.sleep(5)
    print("패티 조리 완료")
    return {}

def g_bun(state: RestaurantState) -> dict:
    print("번 조리 시작")
    time.sleep(3)
    print("번 조리 완료")
    return {}

def g_burger_set(state: RestaurantState) -> dict:
    print("버거 세트 조립 시작")
    time.sleep(2)
    print("버거 세트 조립 완료")
    return {}

def g_burger_complete(state: RestaurantState) -> dict:
    print("버거 완성")
    return {'menu_complete': ['burger']}

def g_fry(state: RestaurantState) -> dict:
    print("감자튀김 조리 시작")
    time.sleep(3)
    print("감자튀김 조리 완료")
    return {}

def g_fries_set(state: RestaurantState) -> dict:
    print("감자튀김 세트 준비 시작")
    time.sleep(1)
    print("감자튀김 세트 준비 완료")
    return {}

def g_fries_complete(state: RestaurantState) -> dict:
    print("감자튀김 완성")
    return {'menu_complete': ['fries']}

def g_beverage(state: RestaurantState) -> dict:
    print("음료 준비 시작")
    time.sleep(2)
    print("음료 준비 완료")
    return {'menu_complete': ['beverage']}

In [ ]:
from langgraph.graph import StateGraph, START, END

def is_order_done(state: RestaurantState):
    return set(state.menu_complete) == set(state.menu_ordered)

def dispatch_router(state: RestaurantState):
    routes = []

    if 'burger' in state.menu_ordered:
        routes.append('burger_start')
    if 'fries' in state.menu_ordered:
        routes.append('fries_start')
    if 'beverage' in state.menu_ordered:
        routes.append('beverage')
    return routes if routes else END

def finish_router(state: RestaurantState):
    if is_order_done(state):
        print(f"주문 완료. 완성 메뉴: {sorted(state.menu_complete)}")
    return END

workflow = StateGraph(RestaurantState)

workflow.add_node('burger_start', g_burger_start)
workflow.add_node('fries_start', g_fries_start)
workflow.add_node('patty', g_patty)
workflow.add_node('bun', g_bun)
workflow.add_node('burger_set', g_burger_set)
workflow.add_node('burger_complete', g_burger_complete)
workflow.add_node('fry', g_fry)
workflow.add_node('fries_set', g_fries_set)
workflow.add_node('fries_complete', g_fries_complete)
workflow.add_node('beverage', g_beverage)

workflow.add_conditional_edges(START, dispatch_router)

workflow.add_edge('burger_start', 'patty')
workflow.add_edge('burger_start', 'bun')
workflow.add_edge(['patty', 'bun'], 'burger_set')
workflow.add_edge('burger_set', 'burger_complete')
workflow.add_conditional_edges('burger_complete', finish_router)

workflow.add_edge('fries_start', 'fry')
workflow.add_edge('fry', 'fries_set')
workflow.add_edge('fries_set', 'fries_complete')
workflow.add_conditional_edges('fries_complete', finish_router)

workflow.add_conditional_edges('beverage', finish_router)

app = workflow.compile()

In [105]:
app.invoke(RestaurantState(menu_ordered=['burger', 'beverage', 'fries']))

음료 준비 시작
버거 준비 시작
감자튀김 준비 시작
음료 준비 완료
번 조리 시작
감자튀김 조리 시작
패티 조리 시작
감자튀김 조리 완료번 조리 완료

패티 조리 완료
버거 세트 조립 시작
감자튀김 세트 준비 시작
감자튀김 세트 준비 완료
버거 세트 조립 완료
버거 완성
감자튀김 완성


{'menu_ordered': ['burger', 'beverage', 'fries'],
 'menu_complete': ['beverage', 'burger', 'fries']}

In [103]:
for chunk in app.stream(RestaurantState(menu_ordered=['burger', 'beverage'])):
    print(chunk)

음료 준비 시작
버거 준비 시작
{'burger_start': None}
음료 준비 완료
{'beverage': {'menu_complete': ['beverage']}}
번 조리 시작
패티 조리 시작
번 조리 완료
{'bun': None}
패티 조리 완료
{'patty': None}
버거 세트 조립 시작
버거 세트 조립 완료
{'burger_set': None}
버거 완성
주문 완료. 완성 메뉴: ['beverage', 'burger']
{'burger_complete': {'menu_complete': ['burger']}}
